# Spam VI: fastText, embeddings with sub-words

TF-IDF has two blind spots:

1. **No similarity.** `cash`, `money` and `prize` are as unrelated as `cash` and `banana`.
2. **No out-of-vocabulary (OOV) handling.** `fr33`, `fre3` and `freee` are unknown words: their evidence is lost.

**fastText** (Bojanowski et al., 2017; Joulin et al., 2017) addresses both. Every word is represented by the *sum of the vectors
of its character n-grams* (`<fr`, `fre`, `ree`, `ee>`, ...), so a misspelt word still shares most n-grams with the correct one.
A message vector is the *average* of its word (and word-bigram) vectors, followed by a linear classifier. It trains in seconds
on a CPU.

We use it two ways: (A) **unsupervised embeddings** fed to a classifier, and (B) the **supervised** fastText classifier.

> **Note (fastText 0.9.3 + NumPy 2).** `model.predict("text")` with a single string fails with NumPy 2. Always pass a *list* of
> strings: `model.predict([text])`.

In [ ]:
import re
import tempfile
from pathlib import Path

import fasttext
import numpy as np
from matplotlib import pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

import spamlib as sl

plt.rcParams["figure.dpi"] = 100

texts, y = sl.load()
X_train, X_test, y_train, y_test = sl.split(texts, y)


def norm(text):
    """Light normalisation for fastText: lower-case, separate punctuation, keep letters/digits inside words (sub-words need them)."""
    text = re.sub(r"([!£$€%?.,;:()\"'])", r" \1 ", text.lower())
    return re.sub(r"\s+", " ", text).strip()


tmp = tempfile.TemporaryDirectory()
train_path = Path(tmp.name) / "train.txt"
train_path.write_text("\n".join(f"__label__{'spam' if lab else 'ham'} {norm(t)}" for t, lab in zip(X_train, y_train)))
unsup_path = Path(tmp.name) / "unsup.txt"
unsup_path.write_text(
    "\n".join(norm(t) for t in X_train)
)  # *training* text only: never let the test text into the embeddings

## A. Unsupervised skip-gram embeddings

Skip-gram learns word vectors by predicting the words *around* a word. Words used in similar contexts get similar vectors.
With only 4,000 messages the space is small, but the mechanism is visible.

In [ ]:
emb = fasttext.train_unsupervised(
    str(unsup_path), model="skipgram", dim=100, ws=5, epoch=25, minn=2, maxn=5, minCount=2, verbose=0
)
for word in ("free", "prize", "call", "home"):
    print(f"{word:6}", [w for _, w in emb.get_nearest_neighbors(word, k=6)])

### Out-of-vocabulary words get a vector too

`fr33` and `congratulationz` never occur in the training data. Their vectors are assembled from their n-grams and land in the
neighbourhood of the correctly spelt words. A bag-of-words model has *nothing* to say about them.

In [ ]:
for word in ("fr33", "freee", "congratulationz", "txtt"):
    in_vocab = word in emb.words
    print(f"{word:16} in vocabulary: {in_vocab!s:5}  nearest: {[w for _, w in emb.get_nearest_neighbors(word, k=5)]}")

### Sentence vectors + a classifier

`get_sentence_vector` averages the (normalised) word vectors. Any classifier can take it from there.

In [ ]:
def sentence_matrix(model, docs):
    return np.vstack([model.get_sentence_vector(norm(d)) for d in docs])


lr_emb = LogisticRegression(C=10, max_iter=2000).fit(sentence_matrix(emb, X_train), y_train)
score_a = lr_emb.decision_function(sentence_matrix(emb, X_test))
results = {"skip-gram + LR": sl.scores(y_test, (score_a >= 0).astype(int), score_a)}
sl.show(results)

## B. The supervised fastText classifier

Architecture: **embedding table** (words *and* hashed word-bigrams *and* character n-grams) $\to$ **average** $\to$ **linear layer** $\to$ **softmax**.
It is a linear model with a low-rank weight matrix, trained end to end by SGD. `wordNgrams=2` adds bigrams (`"call now"`),
`minn/maxn` the character n-grams.

In [ ]:
clf = fasttext.train_supervised(
    str(train_path), lr=0.5, epoch=30, wordNgrams=2, dim=50, minn=2, maxn=5, loss="softmax", verbose=0, seed=sl.SEED
)


def ft_spam_score(model, docs):
    labels, probs = model.predict([norm(d) for d in docs], k=2)
    return np.array([p[lab.index("__label__spam")] for lab, p in zip(labels, probs)])


p_spam = ft_spam_score(clf, X_test)
results["fastText supervised"] = sl.scores(y_test, (p_spam >= 0.5).astype(int), p_spam)

# baselines on the same split
for name, vec in (
    (
        "TF-IDF words + LR",
        TfidfVectorizer(
            tokenizer=sl.tokenize, token_pattern=None, lowercase=False, ngram_range=(1, 2), min_df=2, sublinear_tf=True
        ),
    ),
    ("TF-IDF chars + LR", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3, sublinear_tf=True)),
):
    base = make_pipeline(vec, LogisticRegression(C=30, max_iter=2000)).fit(X_train, y_train)
    s = base.decision_function(X_test)
    results[name] = sl.scores(y_test, (s >= 0).astype(int), s)
sl.show(results)

On clean text fastText is *on par* with a well-tuned linear model. The point of fastText is not a higher score on this
corpus; it is (i) **speed** at scale, (ii) **dense, meaningful vectors**, and (iii) **robustness to spelling variation**.
Let us check (iii).

## Robustness to character-level manipulation

A spammer knows that filters match words, so they misspell them (`fr33`, `vi@gra`). Two experiments, both applied **only to the
spam** messages of the test set (so we measure **recall**: the fraction of spam still caught):

1. **random noise**: each letter is deleted, doubled, swapped or leet-substituted with a growing probability;
2. **targeted obfuscation**: the 25 words with the highest weight in a linear model (`free`, `txt`, `call`, `claim`...) are
   disguised (`fr33`, `f.r.e.e`, `f r e e`, `fr`).

In [ ]:
models = {
    "TF-IDF words + LR": make_pipeline(
        TfidfVectorizer(
            tokenizer=sl.tokenize, token_pattern=None, lowercase=False, ngram_range=(1, 2), min_df=2, sublinear_tf=True
        ),
        LogisticRegression(C=30, max_iter=2000),
    ).fit(X_train, y_train),
    "TF-IDF chars + LR": make_pipeline(
        TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3, sublinear_tf=True),
        LogisticRegression(C=30, max_iter=2000),
    ).fit(X_train, y_train),
}
spam_test = [t for t, lab in zip(X_test, y_test) if lab == 1]


def recalls(docs):
    out = {name: float(m.predict(docs).mean()) for name, m in models.items()}
    out["fastText supervised"] = float((ft_spam_score(clf, docs) >= 0.5).mean())
    return out


rates = [0.0, 0.05, 0.10, 0.20, 0.30]
curves = {name: [] for name in [*models, "fastText supervised"]}
for rate in rates:
    rng = np.random.default_rng(7)
    for name, r in recalls([sl.typo_noise(t, rate, rng) for t in spam_test]).items():
        curves[name].append(r)

fig, ax = plt.subplots(figsize=(6.5, 4))
for name, vals in curves.items():
    ax.plot(rates, vals, "o-", label=name)
ax.set(xlabel="fraction of letters corrupted", ylabel="spam recall", ylim=(0, 1.02))
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
lr_word = models["TF-IDF words + LR"]
names = np.array(lr_word[0].get_feature_names_out())
trigger = [w for w in names[np.argsort(-lr_word[1].coef_[0])] if w.isalpha()][:25]
print("trigger words:", trigger)

print(f"{'obfuscation':12}" + "".join(f"{n:>22}" for n in recalls(spam_test)))
for mode in ("none", "leet", "vowel", "space", "dots"):
    docs = spam_test if mode == "none" else [sl.obfuscate(t, trigger, mode) for t in spam_test]
    print(f"{mode:12}" + "".join(f"{v:22.3f}" for v in recalls(docs).values()))

**What to conclude.**

* None of the models collapses: spam is **redundant** (many independent cues such as the phone number, `£`, `!`, `call`,
  `txt`), and disguising 25 words leaves most of them intact.
* Sub-words help a little with *random* noise, but a *targeted* manipulation that changes the **tokenisation** (dots, spaces)
  hurts the fastText model the most: robustness depends on the *threat model*, not on the model family.
* The attacker who *optimises* (choosing the few edits that matter, querying the filter) is far stronger: notebook 07.

## Exercises

1. Train the supervised model with `minn=0, maxn=0` (no sub-words). Repeat both experiments. What changes?
2. Set `wordNgrams=1`. Which spam messages are now missed? (Print five.)
3. `epoch=30` and `lr=0.5` were not tuned. Tune them on a validation split, without touching the test set.
4. Defence idea: normalise before classifying (remove dots/spaces between single letters, map leetspeak back). Implement it and
   measure how much recall you win back, then think about how the attacker would respond.